# Multimodal Cancer Classification Challenge 2026 — Baseline

Two-branch ResNet-18 over brightfield (BF) + fluorescence (FL), patient-grouped
stratified 3-fold CV, AUC tracking. End-to-end: profile → train → predict → submit.

**Settings to enable in the right sidebar before running:**
- Accelerator = **GPU T4 x2** (or P100)
- Internet = **On** (needed for ImageNet weights)
- Persistence = **Files only**


In [ ]:
# Imports + environment check
import os, re, json, time, random, glob
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, LeaveOneGroupOut
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L


In [ ]:
# Config -- edit hyperparameters here
DATA_ROOT = Path("/kaggle/input/multimodal-cancer-classification-challenge-2026")
OUT_DIR   = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CV         = "sgkf"   # "sgkf" | "gkf" | "lopo"
N_SPLITS   = 3
SEED       = 1        # seed=1 gives balanced folds (every fold has both classes)
EPOCHS     = 8
BATCH_SIZE = 128
LR         = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4
PRETRAINED = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Normalization stats (computed from 500 random train images on 2026-05-12)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144


In [ ]:
# Dataset: pairs BF + FL images by filename
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

class CellDataset(Dataset):
    def __init__(self, df, bf_dir, fl_dir, bf_transform, fl_transform, paired_transform=None):
        self.df = df.reset_index(drop=True)
        self.bf_dir = Path(bf_dir); self.fl_dir = Path(fl_dir)
        self.bf_transform = bf_transform; self.fl_transform = fl_transform
        self.paired_transform = paired_transform

    def __len__(self): return len(self.df)

    @staticmethod
    def _load(p): return Image.open(p).convert("L")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_transform(self._load(self.bf_dir / name))
        fl = self.fl_transform(self._load(self.fl_dir / name))
        if self.paired_transform is not None:
            bf, fl = self.paired_transform(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        return {"bf": bf, "fl": fl, "label": label, "name": name}

def load_train_df(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df


In [ ]:
# Patient-grouped CV. With only 5 cancer + 7 healthy patients in train, we use
# StratifiedGroupKFold so each fold has both classes. seed=1 is verified safe.
def stratified_patient_kfold(df, n_splits=3, seed=1, strict=True):
    skgf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = df["Diagnosis"].to_numpy(); groups = df["patient_id"].to_numpy()
    splits = list(skgf.split(df, y=y, groups=groups))
    if strict:
        for f,(_, va) in enumerate(splits):
            if len(np.unique(y[va])) < 2:
                raise ValueError(f"Fold {f} has only one class. Try a different SEED.")
    return splits

def leave_one_patient_out(df):
    return list(LeaveOneGroupOut().split(df, groups=df["patient_id"].to_numpy()))

def get_splits(df, cv, n_splits, seed):
    if cv == "sgkf":  return stratified_patient_kfold(df, n_splits, seed)
    if cv == "gkf":   return list(GroupKFold(n_splits=n_splits).split(df, groups=df["patient_id"]))
    if cv == "lopo":  return leave_one_patient_out(df)
    raise ValueError(cv)

def summarize_split(df, tr, va):
    trd, vad = df.iloc[tr], df.iloc[va]
    return (f"train: {len(tr):>6} cells, {trd['patient_id'].nunique():>2}p, "
            f"pos {trd['Diagnosis'].mean():.3f} | "
            f"val: {len(va):>6} cells, {vad['patient_id'].nunique():>2}p, "
            f"pos {vad['Diagnosis'].mean():.3f} | "
            f"val pats: {sorted(vad['patient_id'].unique().tolist())}")


In [ ]:
# Augmentations. BF and FL get the SAME geometric augs (flip, rotate) so they
# stay aligned. Photometric jitter can differ per modality.
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

def train_modality_transform(modality):
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([T.ColorJitter(brightness=0.15, contrast=0.15), norm])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

class PairedGeoAug:
    def __init__(self, p_hflip=0.5, p_vflip=0.5, max_rot=30):
        self.p_hflip, self.p_vflip, self.max_rot = p_hflip, p_vflip, max_rot
    def __call__(self, bf, fl):
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        a = random.uniform(-self.max_rot, self.max_rot)
        return TF.rotate(bf, a), TF.rotate(fl, a)


In [ ]:
# Two-branch ResNet-18: one branch per modality, late fusion via a small head.
def _make_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    # 3-channel -> 1-channel input; average pretrained RGB weights
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features
    net.fc = nn.Identity()
    return net, fd

class MultimodalClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=0.3):
        super().__init__()
        self.bf_branch, fd = _make_branch(pretrained)
        self.fl_branch, _  = _make_branch(pretrained)
        self.head = nn.Sequential(
            nn.Linear(fd*2, 256), nn.ReLU(True),
            nn.Dropout(dropout), nn.Linear(256, 1),
        )
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)


In [ ]:
# Quick profile: confirm patient counts and class balance match what we expect.
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
per_pat = df_train.groupby("patient_id").agg(
    n_cells=("Name","size"), label=("Diagnosis","first"),
).reset_index()
print(per_pat.to_string(index=False))
print(f"\nTest:  {len(df_test)} cells")


In [ ]:
# Training loop with AMP, BCE+pos_weight, AUC tracking, best-AUC checkpoint.
def run_epoch(model, loader, optimizer, scaler, criterion, train):
    model.train(train)
    losses, ys, ps = [], [], []
    for batch in loader:
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        y  = batch["label"].float().to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); optimizer.step()
        losses.append(loss.item())
        ys.append(y.detach().cpu().numpy())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
    ys = np.concatenate(ys); ps = np.concatenate(ps)
    auc = roc_auc_score(ys, ps) if len(np.unique(ys)) > 1 else float("nan")
    return float(np.mean(losses)), auc, ys, ps


def train_fold(df, train_idx, val_idx, fold):
    bf_dir = DATA_ROOT / "BF" / "train"; fl_dir = DATA_ROOT / "FL" / "train"
    train_ds = CellDataset(df.iloc[train_idx], bf_dir, fl_dir,
                           train_modality_transform("bf"), train_modality_transform("fl"),
                           paired_transform=PairedGeoAug())
    val_ds   = CellDataset(df.iloc[val_idx], bf_dir, fl_dir,
                           eval_modality_transform("bf"), eval_modality_transform("fl"))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                              persistent_workers=NUM_WORKERS>0)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=NUM_WORKERS>0)

    model = MultimodalClassifier(pretrained=PRETRAINED).to(DEVICE)
    pos = (df.iloc[train_idx]["Diagnosis"]==1).sum()
    neg = (df.iloc[train_idx]["Diagnosis"]==0).sum()
    pos_weight = torch.tensor(neg/max(pos,1), device=DEVICE)
    print(f"  pos_weight = {pos_weight.item():.3f}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda") if DEVICE=="cuda" else None

    history = []
    best_auc = -1.0
    ckpt_path = OUT_DIR / f"fold{fold}_best.pt"
    for ep in range(EPOCHS):
        t0 = time.time()
        tr_loss, tr_auc, _, _ = run_epoch(model, train_loader, optimizer, scaler, criterion, True)
        with torch.no_grad():
            va_loss, va_auc, vy, vp = run_epoch(model, val_loader, None, None, criterion, False)
        sched.step()
        dt = time.time() - t0
        print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} "
              f"| va_loss {va_loss:.4f} va_auc {va_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc,
                        "va_loss": va_loss, "va_auc": va_auc, "time": dt})
        if va_auc > best_auc:
            best_auc = va_auc
            torch.save({"model": model.state_dict(), "epoch": ep, "val_auc": va_auc}, ckpt_path)
            oof = pd.DataFrame({"Name": df.iloc[val_idx]["Name"].values,
                                "patient_id": df.iloc[val_idx]["patient_id"].values,
                                "y_true": vy, "y_pred": vp})
            oof.to_csv(OUT_DIR / f"fold{fold}_oof.csv", index=False)
    with open(OUT_DIR / f"fold{fold}_history.json", "w") as f:
        json.dump({"history": history, "best_auc": best_auc}, f, indent=2)
    return best_auc


In [ ]:
# Train all folds in sequence (~10-15 min per fold on T4 with EPOCHS=8)
splits = get_splits(df_train, CV, N_SPLITS, SEED)
print(f"CV={CV}, total folds = {len(splits)}\n")

best_aucs = []
for fold, (tr, va) in enumerate(splits):
    print(f"=== FOLD {fold} ===")
    print("  " + summarize_split(df_train, tr, va))
    best = train_fold(df_train, tr, va, fold)
    best_aucs.append(best)
    print(f"  best AUC fold {fold} = {best:.4f}\n")
print(f"Per-fold best AUC: {[f'{a:.4f}' for a in best_aucs]}")
print(f"Mean: {np.mean(best_aucs):.4f}  Std: {np.std(best_aucs):.4f}")


In [ ]:
# Learning curves (needed for the presentation slides!)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for hp in sorted(glob.glob(str(OUT_DIR / "fold*_history.json"))):
    h = json.load(open(hp))["history"]
    label = Path(hp).stem.replace("_history", "")
    ax[0].plot([e["epoch"] for e in h], [e["va_loss"] for e in h], marker="o", label=label)
    ax[1].plot([e["epoch"] for e in h], [e["va_auc"]  for e in h], marker="o", label=label)
ax[0].set(title="Validation loss",  xlabel="epoch", ylabel="BCE")
ax[1].set(title="Validation AUC",   xlabel="epoch", ylabel="AUC")
ax[1].axhline(0.85, color="red", linestyle="--", alpha=0.5, label="target 0.85")
for a in ax: a.legend(); a.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Out-of-fold (OOF) AUC -- this is the honest leaderboard estimate
oofs = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(str(OUT_DIR / "fold*_oof.csv")))])
print(f"OOF AUC overall: {roc_auc_score(oofs['y_true'], oofs['y_pred']):.4f}  on {len(oofs)} cells")

print("\nPer-patient mean prediction (mean cell score) vs true label:")
pp = oofs.groupby("patient_id").agg(mean_pred=("y_pred","mean"),
                                    median_pred=("y_pred","median"),
                                    label=("y_true","first")).sort_values("mean_pred")
print(pp.to_string())
print(f"\nPatient-level AUC: {roc_auc_score(pp['label'], pp['mean_pred']):.4f}")


In [ ]:
# Predict on test with 4-way flip TTA, average across folds, write submission.csv
def predict_one_ckpt(ckpt_path, loader, tta=True):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model = MultimodalClassifier(pretrained=False).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=DEVICE=="cuda"):
                p = torch.sigmoid(model(bf, fl)).float()
                if tta:
                    for flips in [(True,False),(False,True),(True,True)]:
                        bft, flt = bf, fl
                        if flips[0]: bft, flt = TF.hflip(bft), TF.hflip(flt)
                        if flips[1]: bft, flt = TF.vflip(bft), TF.vflip(flt)
                        p = p + torch.sigmoid(model(bft, flt)).float()
                    p = p / 4
            preds.append(p.cpu().numpy())
    return np.concatenate(preds)

test_ds = CellDataset(df_test, DATA_ROOT/"BF"/"test", DATA_ROOT/"FL"/"test",
                      eval_modality_transform("bf"), eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

ckpts = sorted(glob.glob(str(OUT_DIR / "fold*_best.pt")))
print("Using ckpts:", ckpts)
all_preds = []
for ckpt in ckpts:
    print(f"  - {ckpt}")
    all_preds.append(predict_one_ckpt(ckpt, test_loader, tta=True))
preds = np.mean(all_preds, axis=0)

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote /kaggle/working/submission.csv  (mean pred = {preds.mean():.3f})")
print(sub.head())
!head /kaggle/working/submission.csv
!wc -l /kaggle/working/submission.csv


## Submitting

1. Click **"Save Version"** (top-right) → **"Save & Run All (Commit)"**. This re-runs the entire notebook in a fresh kernel.
2. When the version finishes, click the notebook's three-dot menu → **"View Versions"** → click the latest version → **"Output"** tab.
3. Find `submission.csv` → click **"Submit to Competition"**.

You only get **4 submissions per day**. Use them on:
1. A baseline (this notebook as-is)
2. With/without TTA comparison
3. A change you actually believe in (new model, SSL pretraining, etc.)
4. A safety re-run if one above failed

**Things to try next** (each gets its own notebook fork):
- Stronger backbones: EfficientNet-B0, ConvNeXt-tiny
- Single-modality ablations: BF-only vs FL-only vs both (for the presentation!)
- SSL pretraining (SimCLR or MAE) on train+test images, then fine-tune
- Per-patient aggregation: average per-patient cell predictions, use that as the patient-level prior
